## RFM and Customer segmentation


First, let's create an RFM model:

R — Recency, i.e. days since last purchase 

F — Frequency, i.e. #s of orders

M — Monetary, i.e net revenue


In [ ]:
# Import Python packages
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np

# Get the current credentials
session = get_active_session()
customers = session.table("PRODUCT_ANALYTICS.ANALYTICS.FCT_CUSTOMER_VALUE").to_pandas()

In [ ]:
customers["R_SCORE"] = pd.qcut(
    customers["DAYS_SINCE_LAST_PURCHASE_END_2025"],
    5,
    labels=[5, 4, 3, 2, 1]
)

customers["F_SCORE"] = pd.qcut(
    customers["ORDER_COUNT"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

customers["M_SCORE"] = pd.qcut(
    customers["TOTAL_REVENUE"].rank(method="first"),
    5,
    labels=[1, 2, 3, 4, 5]
)

customers["RFM_SCORE"] = (
    customers["R_SCORE"].astype(str).replace('nan', '0')
    + customers["F_SCORE"].astype(str).replace('nan', '0')
    + customers["M_SCORE"].astype(str).replace('nan', '0')
).astype(int)

In [ ]:
customers["RFM_SCORE"].describe()

We can then turn those into meaningful segments:
- Champions
- Loyal customers
- Potential loyalists
- New customers
- At risk
- Hibernating

Let's use K-means as a segmentation approach and compare the interpretable RFM segmentation against an unsupervised approach.

In [ ]:
import matplotlib.pyplot as plt # for plotting graphs
from sklearn.cluster import KMeans

X = customers[['R_SCORE', 'F_SCORE', 'M_SCORE']].astype(str).replace('nan', '0').astype(int)

# Calculate inertia (sum of squared distances) for different values of k
inertia = []
for k in range(2, 11):
    kmeans = KMeans(n_clusters=k, n_init=10, random_state=42)
    kmeans.fit(X)
    inertia.append(kmeans.inertia_)

# Plot the elbow curve
plt.figure(figsize=(8, 6),dpi=150)
plt.plot(range(2, 11), inertia, marker='o')
plt.xlabel('Number of Clusters (k)')
plt.ylabel('Inertia')
plt.title('Elbow Curve for K-means Clustering')
plt.grid(True)
plt.show()

In [ ]:
# Perform K-means clustering with best K
best_kmeans = KMeans(n_clusters=4, n_init=10, random_state=42)
X['Cluster'] = best_kmeans.fit_predict(X)

# Group by cluster and calculate mean values
cluster_summary = X.groupby('Cluster').agg({
    'R_SCORE': 'mean',
    'F_SCORE': 'mean',
    'M_SCORE': 'mean'
}).reset_index()

In [ ]:
colors = ['#3498db', '#2ecc71', '#f39c12','#C9B1BD']

# Plot the average RFM scores for each cluster
plt.figure(figsize=(10, 8),dpi=150)

# Plot Avg Recency
plt.subplot(3, 1, 1)
bars = plt.bar(cluster_summary.index, cluster_summary['R_SCORE'], color=colors)
plt.xlabel('Cluster')
plt.ylabel('Avg Recency')
plt.title('Average Recency for Each Cluster')

plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(bars, cluster_summary.index, title='Clusters')

# Plot Avg Frequency
plt.subplot(3, 1, 2)
bars = plt.bar(cluster_summary.index, cluster_summary['F_SCORE'], color=colors)
plt.xlabel('Cluster')
plt.ylabel('Avg Frequency')
plt.title('Average Frequency for Each Cluster')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(bars, cluster_summary.index, title='Clusters')

# Plot Avg Monetary
plt.subplot(3, 1, 3)
bars = plt.bar(cluster_summary.index, cluster_summary['M_SCORE'], color=colors)
plt.xlabel('Cluster')
plt.ylabel('Avg Monetary')
plt.title('Average Monetary Value for Each Cluster')
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend(bars, cluster_summary.index, title='Clusters')

plt.tight_layout()
plt.show()

Notice how the customers in each of the segments can be characterized based on the recency, frequency, and monetary values:

- **Cluster 0**: Of all the four clusters, this cluster has the highest recency, frequency, and monetary values. Let’s call the customers in this cluster **champions** (or **power shoppers**).
- **Cluster 2**: This cluster is characterized by moderate recency, frequency, and monetary values. These customers still spend more and purchase more frequently than clusters 2 and 3. Let’s call them **loyal customers**.
- **Cluster 1**: Customers in this cluster tend to spend less. They don’t buy often, and haven’t made a purchase recently either. These are likely inactive or **at-risk** customers.
- **Cluster 3**: This cluster is characterized by high recency and relatively lower frequency and moderate monetary values. So these are **recent customers** who can potentially become long-term customers.
Here are some examples of how you can tailor marketing efforts—to target customers in each segment—to enhance customer engagement and retention:
--------
- For **Champions/Power Shoppers**: Offer personalized special discounts, early access, and other premium perks to make them feel valued and appreciated.
- For **Loyal Customers**: Appreciation campaigns, referral bonuses, and rewards for loyalty.
- For **At-Risk Customers**: Re-engagement efforts that include running discounts or promotions to encourage buying.
- For **Recent Customers**: Targeted campaigns educating them about the brand and discounts on subsequent purchases. 
It’s also helpful to understand what percentage of customers are in the different segments. This will further help streamline marketing efforts and grow your business.

In [ ]:
cluster_counts = X['Cluster'].value_counts()

colors = ['#3498db', '#2ecc71', '#f39c12','#C9B1BD']
# Calculate the total number of customers
total_customers = cluster_counts.sum()

# Calculate the percentage of customers in each cluster
percentage_customers = (cluster_counts / total_customers) * 100

labels = ['Champions(Power Shoppers)','At-risk Customers','Loyal Customers','Recent Customers']

# Create a pie chart
plt.figure(figsize=(8, 8),dpi=200)
plt.pie(percentage_customers, labels=labels, autopct='%1.1f%%', startangle=90, colors=colors)
plt.title('Percentage of Customers in Each Cluster')
plt.legend(cluster_summary['Cluster'], title='Cluster', loc='upper left')

plt.show()

We have quite an even distribution of customers across segments. So we can invest time and effort in retaining existing customers, re-engaging with at-risk customers, and educating recent customers.

In [ ]:
customers.head()

In [ ]:
customers['KMEANS_CLUSTER'] = X['Cluster']
customers.head()

In [ ]:
customers.info()

In [ ]:
session.use_database("PRODUCT_ANALYTICS")

session.write_pandas(
    customers,
    table_name="FCT_CUSTOMER_SEGMENTS",
    auto_create_table=True
)